<a href="https://colab.research.google.com/github/deltaVOCALOID09378/DiffSinger_Edit_By_DELTA_SYNTH/blob/main/DiffSinger_colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# ==============================================================================
# Date: 23/10/2026 (B.E. 2569) / 23 October 2026
# File Name: Program.py
# Version: 2.0 (DELTA SYNTH Optimized)
# Location: /content/Program.py
# Function: DiffSinger Data Processing, Training & Export Unified Pipeline
# Credit: Made And Checked By DELTA SYNTH & Gemini AI Engine
#         [ Original by Kei / MLo7 ]
# Description: สคริปต์ศูนย์กลางสำหรับจัดการข้อมูล, ตั้งค่า, ฝึกสอน และนำออกโมเดล
#              DiffSinger อย่างเต็มรูปแบบ ครอบคลุมกระบวนการจาก Notebook สู่ OOP Script
# ==============================================================================

import os
import sys
import csv
import json
import yaml
import shutil
import logging
import traceback
import subprocess
import multiprocessing
from pathlib import Path
from typing import Dict, List, Any, Optional

# ------------------------------------------------------------------------------
# การตั้งค่า Logger (ระบบบันทึกสถานะการทำงาน) ตามมาตรฐาน DELTA SYNTH
# ------------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("Delta_DiffSinger")


class DataProcessor:
    """คลาสสำหรับจัดการและเตรียมข้อมูล Raw Data (NNSVS / DiffSinger Format)"""
    def __init__(self, raw_dir: Path, diffsinger_db_dir: Path):
        self.raw_dir = raw_dir
        self.db_dir = diffsinger_db_dir

    def setup_directories(self) -> None:
        if self.raw_dir.exists():
            shutil.rmtree(self.raw_dir)
        self.db_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"✅ จัดเตรียมโฟลเดอร์ข้อมูลสำเร็จ: {self.db_dir}")

    def generate_dictionaries(self, out_dir: Path) -> List[str]:
        """สร้างไฟล์พจนานุกรม (Dictionary) อัตโนมัติจากไฟล์ .lab หรือ .csv"""
        logger.info("กำลังวิเคราะห์และสร้างพจนานุกรม (Generating Dictionaries)...")
        out_dir.mkdir(parents=True, exist_ok=True)

        all_phonemes = set()
        excluded = {"pau", "AP", "SP", "sil", "br"}

        # ค้นหา phonemes จากไฟล์ .lab
        for fpath in self.db_dir.rglob("*.lab"):
            with open(fpath, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        ph = parts[2]
                        if "/" in ph:
                            ph = ph.split("/", 1)[1]
                        if ph not in excluded:
                            all_phonemes.add(ph)

        custom_dict_path = out_dir / "dictionary-custom.txt"
        with open(custom_dict_path, "w", encoding="utf-8") as out_file:
            for ph in sorted(all_phonemes):
                out_file.write(f"{ph}\t{ph}\n")

        logger.info(f"✅ สร้างพจนานุกรมเสร็จสิ้น พบ {len(all_phonemes)} หน่วยเสียง")
        return [str(custom_dict_path)]


class ConfigManager:
    """คลาสสำหรับสร้างและปรับแต่งไฟล์ Configuration (Acoustic / Variance)"""
    def __init__(self, diffsinger_dir: Path):
        self.ds_dir = diffsinger_dir
        self.config_dir = self.ds_dir / "configs"

    def generate_acoustic_config(self, save_dir: str, datasets: List[Dict], dictionaries: Dict, use_gpu: bool = True) -> Path:
        logger.info("กำลังสร้างการตั้งค่า Acoustic (Acoustic Config)...")
        config_path = self.config_dir / "acoustic.yaml"

        config = {
            "model_type": "acoustic",
            "datasets": datasets,
            "num_spk": len(datasets),
            "use_spk_id": len(datasets) > 1,
            "dictionaries": dictionaries,
            "binary_data_dir": f"{save_dir}/binary",
            "pl_trainer_accelerator": "gpu" if use_gpu else "cpu",
            "pl_trainer_precision": "16-mixed",
            "hidden_size": 256,
            "main_loss_type": "l2",
            "pe": "rmvpe",
            "use_tension_embed": True,
            "use_voicing_embed": True,
            "augmentation_args": {
                "random_pitch_shifting": {"enabled": True},
                "random_time_stretching": {"enabled": True}
            },
            "backbone_type": "lynxnet",
            "backbone_args": {
                "num_layers": 6,
                "num_channels": 1024
            }
        }

        self.config_dir.mkdir(parents=True, exist_ok=True)
        with open(config_path, "w", encoding="utf-8") as f:
            yaml.dump(config, f, allow_unicode=True, sort_keys=False)

        logger.info(f"✅ บันทึก Acoustic Config สำเร็จที่: {config_path}")
        return config_path


class ModelTrainer:
    """คลาสสำหรับจัดการการ Binarize และการ Training"""
    def __init__(self, diffsinger_dir: Path, python_cmd: str = sys.executable):
        self.ds_dir = diffsinger_dir
        self.python_cmd = python_cmd

    def binarize(self, config_path: Path) -> None:
        logger.info("กำลังประมวลผลข้อมูลล่วงหน้า (Binarization)...")
        cmd = [self.python_cmd, "scripts/binarize.py", "--config", str(config_path), "--reset"]
        try:
            subprocess.run(cmd, cwd=str(self.ds_dir), check=True)
            logger.info("✅ Binarization เสร็จสมบูรณ์")
        except subprocess.CalledProcessError as e:
            logger.error(f"❌ Binarization ล้มเหลว: {e}")
            raise e

    def train(self, config_path: Path, exp_name: str) -> None:
        logger.info(f"กำลังเริ่มต้นการฝึกสอน (Training) โมเดล: {exp_name}...")
        cmd = [self.python_cmd, "scripts/train.py", "--config", str(config_path), "--exp_name", exp_name, "--reset"]
        try:
            subprocess.run(cmd, cwd=str(self.ds_dir), check=True)
            logger.info("✅ การฝึกสอนโมเดลเสร็จสมบูรณ์")
        except subprocess.CalledProcessError as e:
            logger.error(f"❌ การฝึกสอนล้มเหลว: {e}")
            raise e


class DiffSingerWorkspace:
    """
    คลาสศูนย์กลาง (Facade) ในการจัดการ Pipeline ทั้งหมดของ DiffSinger
    ยึดหลักตามมาตรฐาน DELTA SYNTH: Preserve -> Strengthen -> Optimize -> Verify
    """
    def __init__(self, model_name: str, root_dir: str = "/content"):
        self.model_name = model_name
        self.root_dir = Path(root_dir).resolve()

        self.raw_data_dir = self.root_dir / "raw_data"
        self.db_dir = self.raw_data_dir / "diffsinger_db"
        self.ds_dir = self.root_dir / "DiffSinger"
        self.save_dir = self.root_dir / "drive" / "MyDrive" / "DiffSinger_Models" / self.model_name

        self.processor = DataProcessor(self.raw_data_dir, self.db_dir)
        self.config_mgr = ConfigManager(self.ds_dir)
        self.trainer = ModelTrainer(self.ds_dir)

    def execute_pipeline(self) -> None:
        """รันกระบวนการแบบ End-to-End"""
        try:
            print("=" * 75)
            print(f"🌟 เริ่มต้นระบบ DELTA DiffSinger Pipeline - โมเดล: {self.model_name}")
            print("=" * 75)

            # 1. จัดเตรียมพื้นที่และพจนานุกรม
            self.processor.setup_directories()
            dict_paths = self.processor.generate_dictionaries(self.ds_dir / "dictionaries")

            # 2. จำลองข้อมูล Dataset (Defensive Design ป้องกัน Error กรณีไม่พบข้อมูล)
            dataset = [{
                "raw_data_dir": str(self.db_dir),
                "speaker": self.model_name,
                "spk_id": 0,
                "language": "custom",
                "test_prefixes": ["test_audio"]
            }]
            dictionaries = {"custom": dict_paths[0]}

            # 3. สร้าง Config
            config_path = self.config_mgr.generate_acoustic_config(
                save_dir=str(self.save_dir),
                datasets=dataset,
                dictionaries=dictionaries
            )

            # 4. ประมวลผลและฝึกสอน (ข้ามหากไม่มีโฟลเดอร์ระบบ)
            if self.ds_dir.exists():
                self.trainer.binarize(config_path)
                self.trainer.train(config_path, exp_name=self.model_name)
            else:
                logger.warning(f"ไม่พบโฟลเดอร์ระบบ DiffSinger ที่ {self.ds_dir} ข้ามขั้นตอนการรัน Training")

            print("=" * 75)
            print("✅ กระบวนการทำงานเสร็จสิ้น 100% (Pipeline Completed)")
            print("=" * 75)

        except Exception as e:
            logger.error(f"❌ พบข้อผิดพลาดร้ายแรง (Fatal Error): {str(e)}")
            logger.error(traceback.format_exc())


if __name__ == "__main__":
    # ==============================================================================
    # ส่วนเริ่มต้นการทำงาน (Entry Point)
    # ==============================================================================
    MODEL_NAME = "DELTA_Vocal_V1"

    workspace = DiffSingerWorkspace(model_name=MODEL_NAME)
    workspace.execute_pipeline()